# 11 · Interactive visualisation of every type of geodesic dome

The tetrahedral, icosahedral and dodecahedral domes side by side, interactively:

1. **Map viewers**: each dome in a rotatable `ProjectionViewer`, coloured by the faces of its own base
   solid (`colors='base'`). Drag any map to rotate its sphere, or turn all three together with the sliders.
2. **The explorer**: one dome shown three ways at once (3-D sphere, index-grid net and a map centred on a
   vertex), with the vertex's neighbour rings highlighted in all three. Click the net or the map to choose
   the vertex, and watch a neighbourhood that is compact on the sphere split across the seams of the net.
3. **Cell areas** of the three domes at about the same number of points.

**Live figures need `ipympl`** (`pip install ipympl`). Without it the figures are static images, but every
control below still redraws them.

Script version: [`examples/11_interactive_base_polyhedra.py`](../11_interactive_base_polyhedra.py) · Needs: numpy, matplotlib, ipywidgets (optional: `ipympl` for live, rotatable figures)

<sub>SPDX-License-Identifier: AGPL-3.0-or-later · Copyright (C) 2022-2026 Masahiro Takatsuka. See the NOTICE file for attribution terms.</sub>

## Set-up

In [ ]:
import nb_setup  # makes mt.geodesicdome and examples/dome_utils.py importable from a checkout

WIDGET = nb_setup.setup()  # live figures with ipympl, static images otherwise

In [ ]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from base_explorer import COLOURINGS, BaseExplorer, spherical_areas
from IPython.display import display
from nb_setup import LiveFigure

from mt.geodesicdome.grid.geodesicdome import GeodesicDome
from mt.geodesicdome.grid.polyhedra import BASES
from mt.geodesicdome.interactive import ProjectionViewer, unique_mesh
from mt.geodesicdome.interactive.viewer import DEFAULT_PROJECTIONS

## 1 · Map viewers for the three domes

Frequencies are chosen so the three domes have a similar number of points (about 300). Each map is a
`ProjectionViewer`: with live figures, drag on a map to rotate that sphere, and use the keys of
notebook 09 (arrows, `r`, `p`, `g`, `e`) after clicking on it.

In [ ]:
FREQS = {'tetrahedron': 12, 'icosahedron': 5, 'dodecahedron': 3}       # 290, 252 and 272 points
domes = {b: GeodesicDome(f, base=b) for b, f in FREQS.items()}
viewers, lives = {}, {}
for base, dome in domes.items():
    n_points = len(unique_mesh(dome)[0])
    with plt.ioff():                                    # displayed below, once
        viewers[base] = ProjectionViewer(dome, 'Equal Earth', colors='base', figsize=(9, 4.6),
                                         title=f"GeodesicDome({dome.frequency}, base='{base}'): {n_points} points")
    lives[base] = LiveFigure(fig=viewers[base].fig)
    if WIDGET:
        lives[base].show()
    else:
        display(viewers[base].fig)

### Rotate all three together

The sliders put the same point of the sphere at the centre of every map, so you can compare how the
three lattices cover the same region. *Cell area* colours each triangle by its spherical area relative to
the mean: blue is smaller, red larger (white is average).

In [ ]:
def cell_area_colours(dome):
    points, faces, _ = unique_mesh(dome)
    ratio = spherical_areas(points, faces)
    return ratio / ratio.mean()                         # one value per face


VIEW_COLOURS = {'base face': lambda d: ('base', 'viridis', None, None),
                'position': lambda d: ('position', 'viridis', None, None),
                'cell area': lambda d: (cell_area_colours(d), 'RdBu_r', 0.0, 2.0)}

projection = widgets.Dropdown(options=list(DEFAULT_PROJECTIONS), value='Equal Earth', description='projection')
colouring = widgets.Dropdown(options=list(VIEW_COLOURS), value='base face', description='colour by')
lat = widgets.FloatSlider(value=0, min=-90, max=90, step=1, description='centre lat')
lon = widgets.FloatSlider(value=0, min=-180, max=180, step=1, description='centre lon')
roll = widgets.FloatSlider(value=0, min=-180, max=180, step=1, description='roll')
edges = widgets.Checkbox(value=True, description='edges')


def apply_all(projection, colouring, lat, lon, roll, edges):
    for base, viewer in viewers.items():
        colors, cmap, vmin, vmax = VIEW_COLOURS[colouring](domes[base])
        viewer.cmap = plt.get_cmap(cmap)
        viewer.set_colors(colors, vmin=vmin, vmax=vmax)
        viewer.show_edges = edges
        if projection != viewer.projection_name:
            viewer.set_projection(projection)
        viewer.set_view(lat, lon, roll)                 # redraws
        if not WIDGET:
            display(viewer.fig)


out = widgets.interactive_output(apply_all, dict(projection=projection, colouring=colouring,
                                                 lat=lat, lon=lon, roll=roll, edges=edges))
display(widgets.VBox([widgets.HBox([projection, colouring, edges]), widgets.HBox([lat, lon, roll])]), out)

## 2 · The explorer

`BaseExplorer` (in [`examples/base_explorer.py`](../base_explorer.py)) draws one dome three ways:

* **left**, the sphere, turned to face the centre vertex (drag to turn it);
* **middle**, the index grid the dome is stored on; every stored copy of a ring vertex is marked, so a
  neighbourhood on a seam appears in two or more places (★ marks the centre and its copies);
* **right**, a map projection centred on the vertex.

With live figures, **click a point on the net or on the map** to make it the centre. The buttons jump to a
corner of the base solid (3 or 5 neighbours), a seam vertex, or an interior one.

In [ ]:
with plt.ioff():
    explorer_fig = plt.figure(figsize=(15, 5.6))
explorer = BaseExplorer(explorer_fig, base='tetrahedron', frequency=4, rings=3)
explorer.connect()                                      # clicks choose the centre (live figures)
explorer_live = LiveFigure(fig=explorer_fig)

summary = widgets.Label(explorer.summary())
explorer.on_change(lambda ex: setattr(summary, 'value', ex.summary()))

base = widgets.ToggleButtons(options=BASES, value='tetrahedron', description='base')
freq = widgets.IntSlider(value=4, min=1, max=12, description='frequency', continuous_update=False)
rings = widgets.IntSlider(value=3, min=0, max=8, description='rings', continuous_update=False)
colour = widgets.Dropdown(options=COLOURINGS, value='base face', description='colour by')
centre = widgets.ToggleButtons(options=['interior', 'seam', 'corner'], value='interior', description='centre on')
static_out = widgets.Output()


def show_static():
    if not WIDGET:
        with static_out:
            static_out.clear_output(wait=True)
            display(explorer_fig)


def on_base(change):
    explorer.set_base(change.new)
    show_static()


def on_freq(change):
    explorer.set_frequency(change.new)
    show_static()


def on_rings(change):
    explorer.set_rings(change.new)
    show_static()


def on_colour(change):
    explorer.set_colouring(change.new)
    show_static()


def on_centre(change):
    explorer.select_kind(change.new)
    show_static()


base.observe(on_base, 'value')
freq.observe(on_freq, 'value')
rings.observe(on_rings, 'value')
colour.observe(on_colour, 'value')
centre.observe(on_centre, 'value')

display(widgets.VBox([base, widgets.HBox([freq, rings, colour]), centre, summary]))
if WIDGET:
    explorer_live.show()
else:
    display(static_out)
    show_static()

### Driving the explorer from code

The same methods work without widgets: `set_base`, `set_frequency`, `set_rings`, `set_colouring`,
`select_kind('corner' | 'seam' | 'interior')`, and `select(vertex)` with a stored vertex or any direction
on the sphere. Here: around a corner of each solid, the first rings have 3, 5 or 5 points instead of 6.

In [ ]:
for b in BASES:
    explorer.set_base(b)
    explorer.select_kind('corner')
    print(explorer.summary())
explorer.set_base(base.value)                       # back to the widget's choice
explorer.select_kind(centre.value)
show_static()

## 3 · Cell areas at a similar size

How evenly do the three lattices cover the sphere? The histograms show each triangle's spherical area
relative to the mean, for domes of about 800 points. The tetrahedral cells vary far more: the triangles
next to its four corners are much smaller than those at the face centres, and the spread does not shrink
as the frequency grows.

In [ ]:
SIZES = {'tetrahedron': 20, 'icosahedron': 9, 'dodecahedron': 5}       # 802, 812 and 752 points
fig, ax = plt.subplots(figsize=(8, 3.6))
bins = np.linspace(0, 2.5, 51)
for b, f in SIZES.items():
    points, faces, _ = unique_mesh(GeodesicDome(f, base=b))
    ratio = spherical_areas(points, faces)
    ratio /= ratio.mean()
    cv = ratio.std()
    ax.hist(ratio, bins=bins, histtype='step', lw=2, density=True,
            label=f'{b} f={f}: {len(points)} points, area CV {100 * cv:.0f}%')
ax.set_xlabel('cell area / mean cell area')
ax.set_ylabel('density')
ax.legend(fontsize=8)
fig.tight_layout()
plt.show() if not WIDGET else display(fig.canvas)